# 回答に出典を付けさせ、その出典を検証する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_rag/rag_citation.ipynb)

NLP クックブック 20-05 ／ RAG (検索拡張生成)

RAG の回答がどの資料に基づいているかを示したい。LLM が付けた出典番号が本当に根拠になっているかも確かめたい。

In [1]:
import re
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(messages, max_new_tokens=120):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def answer_with_citations(question, k=3):
    sims = doc_emb @ embedder.encode("query: " + question, normalize_embeddings=True)
    ids = [int(i) for i in sims.argsort()[::-1][:k]]
    context = "\n".join(f"[{n}] {docs[i]}" for n, i in enumerate(ids, 1))
    prompt = ("以下の資料だけに基づいて質問に答えてください。回答の 1 文ごとに、根拠にした資料の番号を文末の句点の前に付けてください。\n"
              "回答の例: 定休日は月曜日です [1]。\n\n"
              f"# 資料\n{context}\n\n# 質問\n{question}")
    return chat([{"role": "user", "content": prompt}]), ids

question = "何時まで営業していて、支払いには何が使えますか？"
answer, ids = answer_with_citations(question)
print(answer)

クックブック食堂は土日であれば 20 時まで営業しており、支払いには現金と QR コード決済に対応しています [3, 1]。


In [2]:
# 検証: 回答の各文について、(1) 引用番号が実在するか (2) 引用先の資料がその文を含意するか (NLI)
nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

def entailment(premise, hypothesis):
    with torch.no_grad():
        probs = nli(**nli_tok(premise, hypothesis, return_tensors="pt", truncation=True)).logits.softmax(-1)[0]
    return probs[0].item()      # このモデルのラベル 0 が entailment

def verify(answer, ids):
    for sent in [s for s in re.split(r"(?<=。)\s*", answer) if s.strip()]:
        cited = [int(n) for group in re.findall(r"\[([\d,、\s]+)\]", sent) for n in re.findall(r"\d+", group)]   # [1] も [3, 1] も拾う
        claim = re.sub(r"\s*\[[\d,、\s]+\]\s*", "", sent).strip()
        bad = [n for n in cited if not 1 <= n <= len(ids)]
        if not cited:
            print(f"  出典なし | {claim}")
        elif bad:
            print(f"  存在しない番号 {bad} | {claim}")
        else:
            p = entailment(" ".join(docs[ids[n - 1]] for n in cited), claim)     # 引用された資料をつなげたものを前提にする
            print(f"  {cited} 含意 {p:.2f} {'○' if p > 0.5 else '×'} | {claim}")

verify(answer, ids)
print("--- わざと出典を付け替えた回答 ---")
verify("営業時間は土日は 20 時までです [2]。クレジットカードが使えます [3]。", ids)

  [3, 1] 含意 1.00 ○ | クックブック食堂は土日であれば 20 時まで営業しており、支払いには現金と QR コード決済に対応しています。
--- わざと出典を付け替えた回答 ---
  [2] 含意 0.00 × | 営業時間は土日は 20 時までです。
  [3] 含意 0.24 × | クレジットカードが使えます。


- 出典の書き方は、指示だけでなく回答例を 1 つ見せると安定する (例がないと、最後にまとめて「資料 [1] と [3] に基づく」と書かれがち)。
- 出典番号は「LLM がそう書いた」だけで、正しい保証はない。存在しない番号や、内容と合わない番号が付くことがある。
- NLI による検証は、LLM を使わずに「この資料からこの文が言えるか」を機械的に確かめる方法。回答を文に分けて 1 文ずつ調べるのがコツ。
- 複数の資料を組み合わせないと言えない文は、資料を連結して前提にする。NLI モデルは長い前提や数値の比較が苦手な点に注意。